In [ ]:
from google_play_scraper import reviews
import pandas as pd     
from concurrent.futures import ThreadPoolExecutor, as_completed 
from tqdm import tqdm

#Hier legen wir fest welche App wir scrapen wollen
apps= {"BMW": "de.bmw.connected.mobile20.row"}
#Wir entscheiden uns für den englischsprachigen und deutschsprachigen Raum der Länder (USA, Deutschland, Großbritannien, Indien), 
#da diese die Hauptmärkte von BMW darstellen.
regions= [("en","us"), 
          ("de","de"), 
          ("en","gb"), 
          ("en","in")]
#Die Funktion scrape_reviews lädt die Reviews für eine bestimmte App und ein bestimmtes Land, bis keine weiteren Reviews mehr verfügbar sind.
def scrape_reviews(company, app_id):
    all_reviews=[]

    for lang, country in regions:
        continuation_token=None
        while True:
            result, continuation_token = reviews(app_id, lang=lang, country=country, count=200, continuation_token=continuation_token)
            all_reviews.extend(result)
            if not result:
                print(f"No more reviews available for {company} in {country}.")
                break
            print(f"Reviews loaded for {company} in {country}: {len(all_reviews)}")
            if continuation_token is None:
                print(f"Reached end of reviews for {company} in {country}.")
                break
    
    df=pd.DataFrame(all_reviews)
    df['company']=company
    return df
#Hier verwenden wir ThreadPoolExecutor, um die Reviews für alle Apps gleichzeitig zu laden. 
#Sobald alle Reviews geladen sind, werden sie in einem DataFrame zusammengeführt und als CSV-Datei gespeichert.
datasets=[]
with ThreadPoolExecutor(max_workers=8) as executor:
    futures= []
    for company, app_id in apps.items():
        futures.append(executor.submit(scrape_reviews, company, app_id))
    for future in tqdm(as_completed(futures), total=len(futures)):
        datasets.append(future.result())

dataset = pd.concat(datasets)
dataset.to_csv("data/BMW/eng_ger_auto_reviews_dataset.csv", index=False)
print("Dataset saved")
print("Total reviews:", len(dataset))
